In [1]:
from langchain.agents.middleware import before_model
from langchain.messages import SystemMessage


@before_model
def force_korean(state, runtime):
    print("state : ", state)
    return {
        "messages": [SystemMessage(content="모든 답변은 반드시 한국어로 작성하세요")]
        + state["messages"]
    }

In [2]:
from langchain.agents import create_agent

agent = create_agent(model="openai:gpt-5-nano", tools=[], middleware=[force_korean])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain what runtime is."}]}
)

print(result["messages"][-1].content)

state :  {'messages': [HumanMessage(content='Explain what runtime is.', additional_kwargs={}, response_metadata={}, id='839d9d38-569a-445b-9f3d-072e9cefd1b1')]}
런타임(runtime)은 두 가지 의미로 자주 쓰입니다. 둘 다를 함께 이해하면 더 명확해집니다.

- 시간적 의미: 프로그램이 실제로 “실행 중”인 기간 자체를 가리킵니다. 즉, 시작해 실행되다가 종료될 때까지의 그 시간대.

- 실행 환경적 의미: 실행 중에 프로그램이 작동하고 필요한 서비스를 제공하는 환경을 말합니다. 운영체제, 라이브러리, 가상 머신, 인터프리터 등이 이 환경을 구성하고, 메모리 관리, 입출력, 예외 처리, 보안 정책 등의 기능을 제공합니다. 이 환경을 보통 “런타임 환경” 또는 “런타임 시스템”이라고 부릅니다.

보통 다음과 같은 맥락에서 사용됩니다.

- 런타임 환경/런타임 시스템: 특정 언어가 실행되도록 돕는 프로그램 모음. 예를 들어,
  - Python의 런타임: 파이썬 인터프리터가 제공하는 실행 환경.
  - Java의 런타임(JVM): 자바 바이트코드를 실행하고 관리하는 가상 머신.
  - JavaScript의 런타임: 브라우저나 Node.js가 제공하는 실행 환경.

- 런타임 라이브러리: 프로그램이 실행 중에 필요한 기능(입출력, 문자열 처리, 메모리 관리 등)을 제공하는 코드 모음. 컴파일러가 생성한 기계어 코드가 이 라이브러리를 호출해 동작합니다.

- 컴파일 타임과의 구분: 컴파일 타임은 코드가 번역되는 시점이고, 런타임은 번역된 코드가 실제로 실행되면서 동작하는 시점입니다. 일부 타입 검사나 에러도 런타임에 발생할 수 있습니다(런타임 에러).

- 예시 차이점:
  - C/C++: 컴파일 시에 대부분의 코드가 기계어로 변환되지만, 런타임 라이브러리(예: 표준 C 라이브러리)가 실행 중 필요한 기능을 제공합니다.
  - Java: 바

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.messages import HumanMessage

# 금지어 목록
banned_words = ["비밀번호", "주민번호", "욕설"]


# 여기에 미들웨어 함수를 작성하세요
@before_model
def block_before_specific_words(state, runtime):
    print("state : ", state)

    # 마지막 사용자 메시지
    last_msg = state["messages"][-1].content

    # 금지어 검사
    if any(word in last_msg for word in banned_words):
        return {
            "messages": [
                SystemMessage(
                    content="""금지어가 감지되었기 때문에 요청을 처리하지 않습니다. \
                    금지어가 감지되어 요청을 처리하지 않는다고만 짤막하게 말하십시오.
                    """
                ),
            ]
            + state["messages"]
        }

    # 금지어 없으면 그대로 통과
    return state


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[],
    # 여기에 middleware를 추가하세요
    middleware=[block_before_specific_words],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "내 비밀번호는 1234야"}]}
)

print(result["messages"][-1].content)

state :  {'messages': [HumanMessage(content='내 비밀번호는 1234야', additional_kwargs={}, response_metadata={}, id='4d7d2dfc-0ba1-4c0a-8d1b-62f1ca8d2f61')]}
금지어가 감지되어 요청을 처리하지 않습니다.


In [4]:
result["messages"][-2].content

'내 비밀번호는 1234야'

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.messages import SystemMessage


# 여기에 미들웨어 함수를 작성하세요
@before_model
def change_response_style(state, runtime):

    # 마지막 사용자 메시지
    user_text = state["messages"][-1].content

    # 길이 기준으로 스타일 결정
    if len(user_text) <= 20:
        system_prompt = "짧고 간단하게 답하세요."
    else:
        system_prompt = "예시를 포함해 자세히 설명하세요."

    # SystemMessage를 앞에 추가
    return {"messages": [SystemMessage(content=system_prompt), *state["messages"]]}


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[],
    # 여기에 middleware를 추가하세요
    middleware=[change_response_style], 
)

result = agent.invoke({"messages": [{"role": "user", "content": "RAG란?"}]})

print(result["messages"][-1].content)

RAG는 Retrieval-Augmented Generation의 약자다. 질문에 맞는 외부 문서를 검색(retrieve)하고 그 내용을 바탕으로 답을 생성(generate하는) 모델을 말한다.


In [12]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call


@tool
def add_numbers(a: int, b: int) -> str:
    """두 숫자를 더합니다."""
    return str(a + b)


# 여기에 도구 미들웨어를 작성하세요
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise


agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[add_numbers],
    # 여기에 middleware를 추가하세요
    middleware=[log_tool_calls],
)

result = agent.invoke({"messages": [{"role": "user", "content": "3과 5를 더해줘"}]})

print(result["messages"][-1].content)

[TOOL START]  add_numbers 
[TOOL ARGS] {'a': 3, 'b': 5}
[TOOL END]  result : content='8' name='add_numbers' tool_call_id='call_sLLJbKlJZi43YmOtW42kQUA9'
8
